In [121]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [123]:
df = pd.read_csv(r"C:\Users\adity\DS_Projects\Prompt_Engineering_for_DataScience\data\Titanic_train.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [125]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## Setup API Key

In [128]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if OPENROUTER_API_KEY:
    print("API Key Successfully loaded..")
else:
    print("API key not found")

API Key Successfully loaded..


In [130]:
from openai import OpenAI

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter client initialized successfully.")

OpenRouter client initialized successfully.


In [208]:
#model_name = "mistralai/mistral-7b-instruct"
#model_name = "stepfun/step-3.5-flash:free"
#model_name = "openai/gpt-oss-120b"
#model_name = "openai/gpt-oss-20b:free"
#model_name = "google/gemini-3.1-pro-preview"
model_name = "upstage/solar-pro-3:free"
print(f"Selected model: {model_name}")

Selected model: upstage/solar-pro-3:free


In [210]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

outliers = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers[col] = int(((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum())

In [212]:
duplicates = df.duplicated().sum()
skewness = df[numeric_cols].skew().to_dict()

In [214]:
summary = {
    "shape": df.shape,
    "columns": df.columns.tolist(),
    "dtypes": df.dtypes.astype(str).to_dict(),
    "missing_values": df.isnull().sum().to_dict(),
    "describe": df.describe().to_dict(),
    "outliers": outliers,
    "duplicates": int(duplicates),
    "skewness": skewness
}


In [216]:
summary

{'shape': (891, 12),
 'columns': ['PassengerId',
  'Survived',
  'Pclass',
  'Name',
  'Sex',
  'Age',
  'SibSp',
  'Parch',
  'Ticket',
  'Fare',
  'Cabin',
  'Embarked'],
 'dtypes': {'PassengerId': 'int64',
  'Survived': 'int64',
  'Pclass': 'int64',
  'Name': 'object',
  'Sex': 'object',
  'Age': 'float64',
  'SibSp': 'int64',
  'Parch': 'int64',
  'Ticket': 'object',
  'Fare': 'float64',
  'Cabin': 'object',
  'Embarked': 'object'},
 'missing_values': {'PassengerId': 0,
  'Survived': 0,
  'Pclass': 0,
  'Name': 0,
  'Sex': 0,
  'Age': 177,
  'SibSp': 0,
  'Parch': 0,
  'Ticket': 0,
  'Fare': 0,
  'Cabin': 687,
  'Embarked': 2},
 'describe': {'PassengerId': {'count': 891.0,
   'mean': 446.0,
   'std': 257.3538420152301,
   'min': 1.0,
   '25%': 223.5,
   '50%': 446.0,
   '75%': 668.5,
   'max': 891.0},
  'Survived': {'count': 891.0,
   'mean': 0.3838383838383838,
   'std': 0.4865924542648585,
   'min': 0.0,
   '25%': 0.0,
   '50%': 0.0,
   '75%': 1.0,
   'max': 1.0},
  'Pclass': {'c

In [218]:
prompt_text = f"""  You are a Senior Data Scientist,\
Perform a basic data analysis on the Titanic  dataset, \
identifying key features and potential insights. The data is represented \
by the `df` variable.

{summary}


Tell us which features are not important to predict survived and which features are need to drop ,\
Provide a summary of the dataset, including numerical and categorical features, \
any initial observations about distributions, and potential relationships between variables, \
especially focusing on 'Survived' as the target variable,\
Give the summary analysis in Markdown format."""

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": prompt_text}
        ],
        temperature=0.7
    )

    model_response_content = response.choices[0].message.content
    print("\n--- Model's Data Analysis Report ---")
    print(model_response_content)
except Exception as e:
    print(f"An error occurred: {e}")


--- Model's Data Analysis Report ---
# Titanic Dataset – Basic Exploratory Data Analysis (EDA)

**Model:** `df` – 891 rows × 12 columns  
**Target:** `Survived` (0 = No, 1 = Yes)  

---

## 1️⃣ Dataset Overview  

| Feature | Type | Description | Missing Values |
|---|---|---|---|
| **PassengerId** | int64 | Unique identifier for each passenger | 0 |
| **Survived** | int64 | Survival outcome (target) | 0 |
| **Pclass** | int64 | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd) | 0 |
| **Name** | object | Full passenger name | 0 |
| **Sex** | object | Gender (male / female) | 0 |
| **Age** | float64 | Age in years (missing for 177 rows) | 177 (19.9 %) |
| **SibSp** | int64 | # of siblings / spouses aboard | 0 |
| **Parch** | int64 | # of parents / children aboard | 0 |
| **Ticket** | object | Ticket number (highly unique) | 0 |
| **Fare** | float64 | Ticket fare (missing for 0 rows) | 0 |
| **Cabin** | object | Cabin identifier (missing for 687 rows) | 687 (76.9 %) |
| **Embarked** | object | 

In [224]:
with open("Autonomous_Data_Analysis_report(solar-pro-3).md", "w", encoding="utf-8") as f:
    f.write(model_response_content)

In [109]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [111]:
droped_df = df.drop(columns=["PassengerId","Name","Sex","Ticket","Cabin","Embarked"])
droped_df

,Survived,Pclass,Age,SibSp,Parch,Fare
0,0,3,22.0,1,0,7.2500
1,1,1,38.0,1,0,71.2833
2,1,3,26.0,0,0,7.9250
3,1,1,35.0,1,0,53.1000
4,0,3,35.0,0,0,8.0500
...,...,...,...,...,...,...
886,0,2,27.0,0,0,13.0000
887,1,1,19.0,0,0,30.0000
888,0,3,NaN,1,2,23.4500
889,1,1,26.0,0,0,30.0000


In [113]:
droped_df.isnull().sum()

Survived      0
Pclass        0
Age         177
SibSp         0
Parch         0
Fare          0
dtype: int64

In [115]:
droped_df["Age"].fillna(droped_df["Age"].mean(), inplace=True)

In [117]:
droped_df.isnull().sum()

Survived    0
Pclass      0
Age         0
SibSp       0
Parch       0
Fare        0
dtype: int64